In [ ]:
"""
pinn_model.py

Physics-informed heat diffusion constraint for U-Net LST downscaling.

Governing equation:

    ∇ · (k ∇T) = 0

where:
    T = predicted land surface temperature
    k = thermal conductivity
    ∇T = temperature gradient
    ∇ · (k∇T) = heat diffusion residual

The physics loss is the mean squared PDE residual.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class HeatDiffusionPINN(nn.Module):
    """
    Computes the steady-state heat diffusion physics loss
    for predicted LST maps.
    """

    def __init__(self, dx=10.0, dy=10.0):
        """
        Parameters
        ----------
        dx : float
            Physical distance between pixels in the x-direction,
            in meters.

        dy : float
            Physical distance between pixels in the y-direction,
            in meters.
        """

        super().__init__()

        self.dx = dx
        self.dy = dy

    def gradient_x(self, x):
        """
        Computes the spatial derivative in the x-direction
        using a centered finite difference.

        Shape:
            (B, 1, H, W) -> (B, 1, H, W)
        """

        dx = (
            x[:, :, :, 2:]
            - x[:, :, :, :-2]
        ) / (2.0 * self.dx)

        dx = F.pad(
            dx,
            (1, 1, 0, 0),
            mode="replicate"
        )

        return dx

    def gradient_y(self, x):
        """
        Computes the spatial derivative in the y-direction
        using a centered finite difference.

        Shape:
            (B, 1, H, W) -> (B, 1, H, W)
        """

        dy = (
            x[:, :, 2:, :]
            - x[:, :, :-2, :]
        ) / (2.0 * self.dy)

        dy = F.pad(
            dy,
            (0, 0, 1, 1),
            mode="replicate"
        )

        return dy

    def forward(self, predicted_lst, conductivity):
        """
        Calculates the physics-informed loss.

        Parameters
        ----------
        predicted_lst : torch.Tensor
            U-Net predicted LST.

            Shape:
                (B, 1, H, W)

        conductivity : torch.Tensor
            Thermal conductivity map.

            Accepted shapes:
                (B, H, W)
                (B, 1, H, W)

        Returns
        -------
        physics_loss : torch.Tensor
            Mean squared heat diffusion residual.
        """

        # Make conductivity 4-dimensional if necessary.
        if conductivity.ndim == 3:
            conductivity = conductivity.unsqueeze(1)

        # Ensure conductivity has the same spatial dimensions
        # as the predicted LST.
        if conductivity.shape[-2:] != predicted_lst.shape[-2:]:
            conductivity = F.interpolate(
                conductivity,
                size=predicted_lst.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        # Move conductivity to the same device as the prediction.
        conductivity = conductivity.to(
            device=predicted_lst.device,
            dtype=predicted_lst.dtype
        )

        # Prevent zero or negative conductivity values.
        conductivity = torch.clamp(
            conductivity,
            min=1e-6
        )

        # --------------------------------------------------
        # Calculate temperature gradients
        # --------------------------------------------------

        dT_dx = self.gradient_x(
            predicted_lst
        )

        dT_dy = self.gradient_y(
            predicted_lst
        )

        # --------------------------------------------------
        # Calculate heat flux
        #
        # Fourier's law:
        #
        #     q = -k ∇T
        # --------------------------------------------------

        flux_x = (
            -conductivity
            * dT_dx
        )

        flux_y = (
            -conductivity
            * dT_dy
        )

        # --------------------------------------------------
        # Calculate divergence of heat flux
        #
        #     ∇ · q
        #
        # For steady-state heat diffusion:
        #
        #     ∇ · (k ∇T) = 0
        # --------------------------------------------------

        div_q = (
            self.gradient_x(flux_x)
            + self.gradient_y(flux_y)
        )

        # --------------------------------------------------
        # Physics loss
        #
        # We want the PDE residual to approach zero.
        # --------------------------------------------------

        physics_loss = torch.mean(
            div_q ** 2
        )

        return physics_loss


if __name__ == "__main__":

    # Test the PINN module independently.

    batch_size = 2
    height = 64
    width = 64

    predicted_lst = torch.randn(
        batch_size,
        1,
        height,
        width
    )

    conductivity = torch.ones(
        batch_size,
        1,
        height,
        width
    )

    pinn = HeatDiffusionPINN(
        dx=10.0,
        dy=10.0
    )

    loss = pinn(
        predicted_lst,
        conductivity
    )

    print(
        "Predicted LST shape:",
        predicted_lst.shape
    )

    print(
        "Conductivity shape:",
        conductivity.shape
    )

    print(
        "Physics loss:",
        loss.item()
    )